# Libraries

In [1]:
import os
import random

import numpy as np
import pandas as pd

from keras.models import load_model # Model loading Method

import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ( 
    load_img,
    img_to_array,
    ImageDataGenerator
)

from skimage import (
    io,
    feature, 
    color
)

from skimage.filters import (
    roberts, sobel, scharr, prewitt
)

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    cohen_kappa_score
)     # Metrics


2025-04-11 09:14:17.145740: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-11 09:14:17.187993: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-11 09:14:17.189858: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-04-11 09:14:26.627039: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


# Image Pre-Processing Function

In [2]:
## ----------------------------------------------------------------- Custom preprocessing wrapper function -------------------------------------------------------- #
def preprocessing_wrapper(method='canny'):
    def preprocessing_function(image):
        return perform_edge_detection(image, method=method)
    return preprocessing_function

def perform_edge_detection(image,method):
    image = color.rgb2gray(image)    # Convert to grayscale
    
    if method == 'canny':
        image = feature.canny(image, sigma=0.02).astype(np.float32)
    if method == 'sobel':
        image = sobel(image).astype(np.float32)
    if method == 'roberts':
        image = roberts(image).astype(np.float32)
    if method == 'scharr':
        image = scharr(image).astype(np.float32)
    if method == 'prewitt':
        image = prewitt(image).astype(np.float32)
        
    image = np.expand_dims(image, axis=-1)  # Add channel dimension
    image = np.repeat(image, 3, axis=-1)    # Repeat the single channel to create a 3-channel image
    return image

# Predict and Plotting the images

In [3]:
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow.keras.preprocessing import image as keras_image

def predict_single_image(model, image_path, method, target_size=(224, 224)):
    if type(method) == str: 
        # Load and preprocess the single image
        image = keras_image.load_img(image_path, target_size=target_size)
        image_array = keras_image.img_to_array(image)
        
        # Use your edge detection preprocessing
        image_array = perform_edge_detection(image_array, method)  
        
        image_array = np.expand_dims(image_array, axis=0)  # Add batch dimension
        image_array /= 255.0  # Normalize to [0, 1]
    
        # Perform prediction
        prediction_prob = model.predict(image_array)
        prediction_label = (prediction_prob > 0.5).astype(int)
    else:
        image = keras_image.load_img(image_path, target_size=target_size)
        image_array = keras_image.img_to_array(image)
        image_array = np.expand_dims(image_array, axis=0)  # Add batch dimension
        image_array /= 255.0  # Normalize to [0, 1]

        # Perform prediction
        prediction_prob = model.predict(image_array)
        prediction_label = (prediction_prob > 0.5).astype(int)

    return prediction_prob, prediction_label
        
# Function to plot images
def plot_images(image_paths, cols=5):
    n_images = len(image_paths)
    fig = plt.figure(figsize=(10, 5))
    
    for i, img_path in enumerate(image_paths):
        img = Image.open(img_path)
        ax = fig.add_subplot(n_images // cols + 1, cols, i + 1)
        ax.imshow(img)
        ax.set_title(os.path.basename(img_path))
        ax.axis('off')

    plt.tight_layout()
    plt.show()

# Function to plot images with their prediction probabilities
def plot_images_with_predictions(model, image_paths, figname,supply,method,super_title, target_size=(224, 224), cols=5):
    n_images = len(image_paths)
    fig = plt.figure(figsize=(15, 10))
    
    for i, img_path in enumerate(image_paths):
        # Predict the probability for the single image
        prediction_prob, prediction_label = predict_single_image(model, img_path, method, target_size)
        
        # Load the image for display
        img = Image.open(img_path)
        
        # Add the image to the plot
        ax = fig.add_subplot(n_images // cols + 1, cols, i + 1)
        ax.imshow(img)

        if supply=='fc' and prediction_label == 0:
            bbox = dict(facecolor='green', alpha=0.5)
        elif supply=='fc' and prediction_label == 1:
            bbox = dict(facecolor='red', alpha=0.5)
        elif supply=='nfc' and prediction_label == 1:
            bbox = dict(facecolor='green', alpha=0.5)
        else:
            bbox = dict(facecolor='red', alpha=0.5)
        
        # ax.set_title(f'{os.path.basename(img_path)}\nFC Prob: {1-prediction_prob[0][0]:.4f}\nNon-FC Prob: {prediction_prob[0][0]:.4f}')
        ax.set_title(f'FC Prob: {1-prediction_prob[0][0]:.4f}\nNon-FC Prob: {prediction_prob[0][0]:.4f}',
                     #bbox=dict(facecolor='yellow', alpha=0.5), 
                     bbox=bbox,
                     fontsize=10
                    )
        
        ax.axis('off')
        # Add a super title for the entire figure
    fig.suptitle(f'{super_title}', fontsize=16)
    # plt.tight_layout(rect=[0, 0, 1, 0.96]) 
    plt.tight_layout()
    plt.savefig(figname,dpi=300)
    #plt.show()
    plt.close(fig) 

# Setting Parameters

In [4]:
# seed used in training
seed = 42

# Set the image size and batch size
image_size = (224, 224)  # Inception V3 input size
batch_size = 32

# Image Loading

In [5]:
dataset_dir = '/data'

# Prepare data using ImageDataGenerator
datagen = ImageDataGenerator(rescale=1.0/255.0)

# Create the image generator
data_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='binary',
    # subset='validation',
    seed = seed,
    shuffle=False
)

Found 15032 images belonging to 2 classes.


# Image to Label Mapping

In [6]:
filepaths = data_generator.filepaths
labels = data_generator.labels

# Combine filepaths and labels into a list of tuples
filepaths_to_labels_pairing = list(zip(filepaths, labels))

file_path_to_label_df = pd.DataFrame(filepaths_to_labels_pairing,columns = ['path', 'label'])
file_path_to_label_df

,path,label
0,/data/FC/100.png,0
1,/data/FC/1001.png,0
2,/data/FC/10016.png,0
3,/data/FC/10017.png,0
4,/data/FC/10018.png,0
...,...,...
15027,/data/NON FC/995.png,1
15028,/data/NON FC/9950.png,1
15029,/data/NON FC/9951.png,1
15030,/data/NON FC/9953.png,1


# Sample Images

## For Fairy Circle Images

In [7]:
###------------------------------------------FAIRY-CIRCLES-------------------------------------#
# # 7724 10309 23832 23538 8994
# # Define the list of identifiers
identifiers = '7724 10309 23832 23538 8994'.split()
identifiers = [ identifier + '.png' for identifier in identifiers ]

sample_fc = file_path_to_label_df[
    #(
        file_path_to_label_df['path'].apply(lambda x: os.path.basename(x) in identifiers)
    #) #&
    # (file_path_to_label_df['label'] == 0  )
]
sample_fc

,path,label
130,/data/FC/10309.png,0
2690,/data/FC/23538.png,0
2735,/data/FC/23832.png,0
6164,/data/FC/7724.png,0
6960,/data/FC/8994.png,0


In [8]:
# # Filter the list based on the presence of any identifier in the image path
sample_fc = sample_fc['path'].to_list()
print("sample_fc\n\t",sample_fc)



sample_fc
	 ['/data/FC/10309.png', '/data/FC/23538.png', '/data/FC/23832.png', '/data/FC/7724.png', '/data/FC/8994.png']


## For Non-Fairy Circle Images

In [9]:
###---------------------------------------NON-FAIRY-CIRCLES-------------------------------------#

# # 33 5027 41807 63144 65649 61692
identifiers = '33 5027 41807 63144 65649'.split()
identifiers = [ identifier + '.png' for identifier in identifiers ]
sample_nfc = file_path_to_label_df[
    ( file_path_to_label_df['path'].apply(lambda x: os.path.basename(x) in identifiers) ) &
    (file_path_to_label_df['label'] ==1  )
]
sample_nfc


,path,label
9440,/data/NON FC/33.png,1
9815,/data/NON FC/41807.png,1
11421,/data/NON FC/5027.png,1
12673,/data/NON FC/63144.png,1
13598,/data/NON FC/65649.png,1


In [10]:
sample_nfc = sample_nfc['path'].to_list()

print("\nsample_nfc\n\t",sample_nfc)


sample_nfc
	 ['/data/NON FC/33.png', '/data/NON FC/41807.png', '/data/NON FC/5027.png', '/data/NON FC/63144.png', '/data/NON FC/65649.png']


# RGB Based Models

## Model Paths

In [11]:
alexnet_model_path   = '/app/model-weights/RGB-Based-CNN/alexnet_results/alexnet_best_weights.tf' # Load your trained CNN model
resnet_model_path    = '/app/model-weights/RGB-Based-CNN/resnet_results/resnet_best_weights.tf' # Load your trained CNN model
inception_model_path = '/app/model-weights/RGB-Based-CNN/inceptionv3_results/inception_best_weights.tf'
vgg_model_path       = '/app/model-weights/RGB-Based-CNN/vgg16_results/vgg_best_weights.tf'

In [12]:
rgb_model_to_weights_path = dict()
rgb_model_to_weights_path['VGG16'] = vgg_model_path
rgb_model_to_weights_path['InceptionV3'] = inception_model_path
rgb_model_to_weights_path['AlexNet'] = alexnet_model_path
rgb_model_to_weights_path['ReNnet50'] = resnet_model_path
rgb_model_to_weights_path

{'VGG16': '/app/model-weights/RGB-Based-CNN/vgg16_results/vgg_best_weights.tf',
 'InceptionV3': '/app/model-weights/RGB-Based-CNN/inceptionv3_results/inception_best_weights.tf',
 'AlexNet': '/app/model-weights/RGB-Based-CNN/alexnet_results/alexnet_best_weights.tf',
 'ReNnet50': '/app/model-weights/RGB-Based-CNN/resnet_results/resnet_best_weights.tf'}

## FOR **FAIRY CIRCLE** IMAGES

In [14]:
cwd = os.getcwd()
os.makedirs('RGB_SAMPLE_PREDICTIONS',exist_ok=True)
path_to_rgb_sample_preds = os.path.join(cwd,'RGB_SAMPLE_PREDICTIONS')
path_to_rgb_sample_preds

'/app/RGB_SAMPLE_PREDICTIONS'

In [15]:
##--------------------------------------------------------Setting-Paths-of-SAVED-MODELS-SAMPLE-IMAGES-PATHS

model_to_weights_path = rgb_model_to_weights_path
image_paths =  sample_fc

supply = 'fc'

method = 0 

super_title = 'Fairy Circle Predictions'
path_to_save = path_to_rgb_sample_preds
##---------------------------------------------------------------Plotting-predictions-------------------------------#

models_metrics_map = dict()

for cnn_algo,model_path in model_to_weights_path.items():
    
    loaded_cnn_model = load_model(model_path)

    # model_name_list = cnn_algo.split('_')#[:2]
    # method = model_name_list[1]
    figname = supply.upper()+'_'+cnn_algo + '.png'
    figname = os.path.join(path_to_save,figname)
    print(cnn_algo)
    
    plot_images_with_predictions(
        model=loaded_cnn_model, 
        image_paths=image_paths,
        figname=figname, 
        supply=supply,
        method=method,
        super_title=cnn_algo + ' '+ super_title
    )

VGG16
1/1 [==============================] - 0s 105ms/step
InceptionV3
1/1 [==============================] - 0s 102ms/step
AlexNet
1/1 [==============================] - 0s 56ms/step
ReNnet50
1/1 [==============================] - 0s 150ms/step


## FOR **NON-FAIRY CIRCLE** IMAGES

In [16]:
##--------------------------------------------------------Setting-Paths-of-SAVED-MODELS-SAMPLE-IMAGES-PATHS

model_to_weights_path = rgb_model_to_weights_path

# setting non fairy circle images paths 
image_paths =  sample_nfc

# Supplying non fairy circle
supply = 'nfc'

# 0 - FOR RGB 
method = 0 

# save directory
path_to_save = path_to_rgb_sample_preds

# super title
super_title = 'Non-Fairy Circle Predictions'


##---------------------------------------------------------------Plotting-predictions-------------------------------#

models_metrics_map = dict()

for cnn_algo,model_path in model_to_weights_path.items():
    
    # model_dir_contents = os.listdir(model_to_weights_path[cnn_algo])
    
    # weights_file = [file for file in model_dir_contents if file.endswith('.tf')][0]
    
    # weights_file_path = os.path.join(model_to_weights_path[cnn_algo],weights_file)
    
    loaded_cnn_model = load_model(model_path)

    # model_name_list = cnn_algo.split('_')#[:2]
    # method = model_name_list[1]
    
    figname = supply.upper()+'_'+cnn_algo + '.png'
    figname = os.path.join(path_to_save,figname)
    print(cnn_algo)
    
    plot_images_with_predictions(
        model=loaded_cnn_model, 
        image_paths=image_paths,
        figname=figname, 
        supply=supply,
        method=method,
        super_title=cnn_algo+' '+super_title
    )

VGG16
1/1 [==============================] - 0s 87ms/step
InceptionV3
1/1 [==============================] - 0s 98ms/step
AlexNet
1/1 [==============================] - 0s 57ms/step
ReNnet50
1/1 [==============================] - 0s 173ms/step


# EDGE Based Models

In [17]:
cwd = os.getcwd()
os.makedirs('EDGE-Based-CNN_SAMPLE_PREDICTIONS',exist_ok=True)
path_to_edge_sample_preds = os.path.join(cwd,'EDGE-Based-CNN_SAMPLE_PREDICTIONS')
path_to_edge_sample_preds

'/app/EDGE-Based-CNN_SAMPLE_PREDICTIONS'

## Model Paths

In [20]:
edge_based_weights_dir = '/external/model-weights/EDGE-Based-CNN'

all_files = os.listdir(edge_based_weights_dir)
substrings = ['canny', 
              'sobel','roberts']
weights_dirs = [
    file for file in all_files if any(sub in file for sub in substrings) and file.endswith('.py') !=True and file.endswith('results') ==True
]
weights_dirs

['alexnet_canny_results',
 'alexnet_roberts_results',
 'alexnet_sobel_results',
 'inceptionv3_canny_results',
 'inceptionv3_roberts_results',
 'inceptionv3_sobel_results',
 'resnet50_canny_results',
 'resnet50_roberts_results',
 'resnet50_sobel_results',
 'vgg16_canny_results',
 'vgg16_roberts_results',
 'vgg16_sobel_results']

In [21]:
# cwd = os.getcwd()
edge_model_to_weights_path = dict()
for model in weights_dirs:
    model_path = os.path.join(edge_based_weights_dir,model)
    edge_model_to_weights_path[model] = model_path
    
edge_model_to_weights_path

{'alexnet_canny_results': '/external/model-weights/EDGE-Based-CNN/alexnet_canny_results',
 'alexnet_roberts_results': '/external/model-weights/EDGE-Based-CNN/alexnet_roberts_results',
 'alexnet_sobel_results': '/external/model-weights/EDGE-Based-CNN/alexnet_sobel_results',
 'inceptionv3_canny_results': '/external/model-weights/EDGE-Based-CNN/inceptionv3_canny_results',
 'inceptionv3_roberts_results': '/external/model-weights/EDGE-Based-CNN/inceptionv3_roberts_results',
 'inceptionv3_sobel_results': '/external/model-weights/EDGE-Based-CNN/inceptionv3_sobel_results',
 'resnet50_canny_results': '/external/model-weights/EDGE-Based-CNN/resnet50_canny_results',
 'resnet50_roberts_results': '/external/model-weights/EDGE-Based-CNN/resnet50_roberts_results',
 'resnet50_sobel_results': '/external/model-weights/EDGE-Based-CNN/resnet50_sobel_results',
 'vgg16_canny_results': '/external/model-weights/EDGE-Based-CNN/vgg16_canny_results',
 'vgg16_roberts_results': '/external/model-weights/EDGE-Based-

## **FAIRY CIRCLE** PREDICTION

In [21]:
##--------------------------------------------------------Setting-Paths-of-SAVED-MODELS-SAMPLE-IMAGES-PATHS

model_to_weights_path = edge_model_to_weights_path
image_paths =  sample_fc
supply = 'fc'

super_title = 'Fairy Circle Predictions'

path_to_save = path_to_edge_sample_preds

##---------------------------------------------------------------Plotting-predictions-------------------------------#

models_metrics_map = dict()

for cnn_algo,model_path in model_to_weights_path.items():
    
    model_dir_contents = os.listdir(model_to_weights_path[cnn_algo])
    
    weights_file = [file for file in model_dir_contents if file.endswith('.tf')][0]
    
    weights_file_path = os.path.join(model_to_weights_path[cnn_algo],weights_file)
    
    loaded_cnn_model = load_model(weights_file_path)

    model_name_list = cnn_algo.split('_') #[:2]
    method = model_name_list[1].lower()
    
    figname = supply.upper()+'_'+cnn_algo + '.png'
    figname = os.path.join(path_to_save,figname)
    print(cnn_algo)
    
    plot_images_with_predictions(
        model=loaded_cnn_model, 
        image_paths=image_paths,
        figname=figname, 
        supply=supply,
        method=method,
        super_title=model_name_list[1]+' '+model_name_list[0] + ' '+ super_title
    )
    

alexnet_canny_results
1/1 [==============================] - 0s 61ms/step
alexnet_roberts_results
1/1 [==============================] - 0s 56ms/step
alexnet_sobel_results
1/1 [==============================] - 0s 57ms/step
inceptionv3_canny_results
1/1 [==============================] - 0s 102ms/step
inceptionv3_roberts_results
1/1 [==============================] - 0s 97ms/step
inceptionv3_sobel_results
1/1 [==============================] - 0s 117ms/step
resnet50_canny_results
1/1 [==============================] - 0s 152ms/step
resnet50_roberts_results
1/1 [==============================] - 0s 155ms/step
resnet50_sobel_results
1/1 [==============================] - 0s 142ms/step
vgg16_canny_results
1/1 [==============================] - 0s 92ms/step
vgg16_roberts_results
1/1 [==============================] - 0s 110ms/step
vgg16_sobel_results
1/1 [==============================] - 0s 90ms/step


## **NON FAIRY CIRCLE** PREDICTION

In [22]:
##--------------------------------------------------------Setting-Paths-of-SAVED-MODELS-SAMPLE-IMAGES-PATHS

model_to_weights_path = edge_model_to_weights_path
image_paths =  sample_nfc
supply = 'nfc'

super_title = 'Non-Fairy Circle Predictions'

path_to_save = path_to_edge_sample_preds

##---------------------------------------------------------------Plotting-predictions-------------------------------#

models_metrics_map = dict()

for cnn_algo,model_path in model_to_weights_path.items():
    
    model_dir_contents = os.listdir(model_to_weights_path[cnn_algo])
    
    weights_file = [file for file in model_dir_contents if file.endswith('.tf')][0]
    
    weights_file_path = os.path.join(model_to_weights_path[cnn_algo],weights_file)
    
    loaded_cnn_model = load_model(weights_file_path)

    model_name_list = cnn_algo.split('_')#[:2]
    method = model_name_list[1].lower()
    
    figname = supply.upper()+'_'+cnn_algo + '.png'
    figname = os.path.join(path_to_save,figname)
    print(cnn_algo)
    
    plot_images_with_predictions(
        model=loaded_cnn_model, 
        image_paths=image_paths,
        figname=figname, 
        supply=supply,
        method=method,
        super_title=model_name_list[1]+' '+model_name_list[0] + ' '+ super_title
    )

alexnet_canny_results
1/1 [==============================] - 0s 57ms/step
alexnet_roberts_results
1/1 [==============================] - 0s 58ms/step
alexnet_sobel_results
1/1 [==============================] - 0s 57ms/step
inceptionv3_canny_results
1/1 [==============================] - 0s 93ms/step
inceptionv3_roberts_results
1/1 [==============================] - 0s 106ms/step
inceptionv3_sobel_results
1/1 [==============================] - 0s 95ms/step
resnet50_canny_results
1/1 [==============================] - 0s 143ms/step
resnet50_roberts_results
1/1 [==============================] - 0s 158ms/step
resnet50_sobel_results
1/1 [==============================] - 0s 144ms/step
vgg16_canny_results
1/1 [==============================] - 0s 93ms/step
vgg16_roberts_results
1/1 [==============================] - 0s 99ms/step
vgg16_sobel_results
1/1 [==============================] - 0s 99ms/step


# External

In [37]:
external_images_dir = '/app/External Images/REGIONS - Australia Namibia/pred_label'
external_file_paths = [ os.path.join(external_images_dir, image_file) for image_file in os.listdir(external_images_dir) ]
external_file_paths

['/app/External Images/REGIONS - Australia Namibia/pred_label/whole_australia_cropped.png',
 '/app/External Images/REGIONS - Australia Namibia/pred_label/Whole_namibia_cropped.png']

In [38]:
external_results_df = pd.DataFrame(columns = ['image','algorithm','filter','fc_prob','nfc_prob','pred_label'])
external_results_df

,image,algorithm,filter,fc_prob,nfc_prob,pred_label


## RGB Based

In [39]:
external_results_df = pd.DataFrame(columns = ['image','algorithm','filter','fc_prob','nfc_prob','pred_label'])
external_results_df

##--------------------------------------------------------Setting-Paths-of-SAVED-MODELS-SAMPLE-IMAGES-PATHS

model_to_weights_path = rgb_model_to_weights_path

# setting non fairy circle images paths 
image_paths =  external_file_paths

# # Supplying non fairy circle
# supply = 'nfc'

# 0 - FOR RGB 
method = 0 

# save directory
# path_to_save = path_to_rgb_sample_preds

# super title
# super_title = 'Non-Fairy Circle Predictions'


##---------------------------------------------------------------Plotting-predictions-------------------------------#

models_metrics_map = dict()

for cnn_algo,model_path in model_to_weights_path.items():
    loaded_cnn_model = load_model( model_path )
    print(10 * '****', cnn_algo, 10 * '****')
    for image_path in external_file_paths:
        image_file = os.path.basename(image_path)
        
        pred_prob, pred_label = predict_single_image(loaded_cnn_model, image_path, method)
        print(f'\t{image_file}')
        print(f'\t\tPrediction_prob : {pred_prob[0]} Prediction_label : {pred_label[0]}')
        external_results_df.loc[len(external_results_df)] = {
            'image':image_file,
            'algorithm': cnn_algo,
            'filter': 'No',
            'fc_prob': 1 - pred_prob[0][0],
            'nfc_prob': pred_prob[0][0],
            'pred_label': pred_label[0][0]
        }
    print(20 * '****')
    print()

**************************************** VGG16 ****************************************
1/1 [==============================] - 0s 207ms/step
	whole_australia_cropped.png
		Prediction_prob : [0.9883081] Prediction_label : [1]
1/1 [==============================] - 0s 94ms/step
	Whole_namibia_cropped.png
		Prediction_prob : [0.00202396] Prediction_label : [0]
********************************************************************************

**************************************** InceptionV3 ****************************************
1/1 [==============================] - 1s 1s/step
	whole_australia_cropped.png
		Prediction_prob : [0.9997565] Prediction_label : [1]
1/1 [==============================] - 0s 109ms/step
	Whole_namibia_cropped.png
		Prediction_prob : [0.15750597] Prediction_label : [0]
********************************************************************************

**************************************** AlexNet ****************************************
1/1 [=================

In [40]:
external_results_df

,image,algorithm,filter,fc_prob,nfc_prob,pred_label
0,whole_australia_cropped.png,VGG16,No,0.011692,9.883081e-01,1
1,Whole_namibia_cropped.png,VGG16,No,0.997976,2.023962e-03,0
2,whole_australia_cropped.png,InceptionV3,No,0.000243,9.997565e-01,1
3,Whole_namibia_cropped.png,InceptionV3,No,0.842494,1.575060e-01,0
4,whole_australia_cropped.png,AlexNet,No,0.102218,8.977823e-01,1
5,Whole_namibia_cropped.png,AlexNet,No,0.999999,6.116675e-07,0
6,whole_australia_cropped.png,ReNnet50,No,0.399697,6.003033e-01,1
7,Whole_namibia_cropped.png,ReNnet50,No,0.873516,1.264841e-01,0


## Edge Based

In [41]:
##--------------------------------------------------------Setting-Paths-of-SAVED-MODELS-SAMPLE-IMAGES-PATHS

model_to_weights_path = edge_model_to_weights_path

# setting non fairy circle images paths 
image_paths =  external_file_paths

# # Supplying non fairy circle
# supply = 'nfc'

# save directory
# path_to_save = path_to_rgb_sample_preds

# super title
# super_title = 'Non-Fairy Circle Predictions'


##---------------------------------------------------------------Plotting-predictions-------------------------------#

for cnn_algo,model_path in model_to_weights_path.items():
    
    model_dir_contents = os.listdir(model_to_weights_path[cnn_algo])
    weights_file = [file for file in model_dir_contents if file.endswith('.tf')][0]
    weights_file_path = os.path.join(model_to_weights_path[cnn_algo],weights_file)
    
    loaded_cnn_model = load_model(weights_file_path)

    model_name_list = cnn_algo.split('_')#[:2]
    method = model_name_list[1].lower()
    
    print(cnn_algo)
    
    print(10 * '****', cnn_algo, 10 * '****')
    for image_path in external_file_paths:
        image_file = os.path.basename(image_path)
        
        pred_prob, pred_label = predict_single_image(loaded_cnn_model, image_path, method)
        print(f'\t{image_file}')
        print(f'\t\tPrediction_prob : {pred_prob[0]} Prediction_label : {pred_label[0]}')
        
        external_results_df.loc[len(external_results_df)] = {
            'image':image_file,
            'algorithm': cnn_algo,
            'filter': method,
            'fc_prob': 1 - pred_prob[0][0],
            'nfc_prob': pred_prob[0][0],
            'pred_label': pred_label[0][0]
        }
        
    print(20 * '****')
    print()

alexnet_canny_results
**************************************** alexnet_canny_results ****************************************
1/1 [==============================] - 0s 117ms/step
	whole_australia_cropped.png
		Prediction_prob : [0.00259138] Prediction_label : [0]
1/1 [==============================] - 0s 55ms/step
	Whole_namibia_cropped.png
		Prediction_prob : [0.00260272] Prediction_label : [0]
********************************************************************************

alexnet_roberts_results
**************************************** alexnet_roberts_results ****************************************
1/1 [==============================] - 0s 110ms/step
	whole_australia_cropped.png
		Prediction_prob : [0.9963388] Prediction_label : [1]
1/1 [==============================] - 0s 57ms/step
	Whole_namibia_cropped.png
		Prediction_prob : [0.01640783] Prediction_label : [0]
********************************************************************************

alexnet_sobel_results
*************

In [43]:
external_results_df[ external_results_df['filter']!='No' ]

,image,algorithm,filter,fc_prob,nfc_prob,pred_label
8,whole_australia_cropped.png,alexnet_canny_results,canny,0.997409,0.002591,0
9,Whole_namibia_cropped.png,alexnet_canny_results,canny,0.997397,0.002603,0
10,whole_australia_cropped.png,alexnet_roberts_results,roberts,0.003661,0.996339,1
11,Whole_namibia_cropped.png,alexnet_roberts_results,roberts,0.983592,0.016408,0
12,whole_australia_cropped.png,alexnet_sobel_results,sobel,0.023535,0.976465,1
13,Whole_namibia_cropped.png,alexnet_sobel_results,sobel,0.984741,0.015259,0
14,whole_australia_cropped.png,inceptionv3_canny_results,canny,0.933234,0.066766,0
15,Whole_namibia_cropped.png,inceptionv3_canny_results,canny,0.918144,0.081856,0
16,whole_australia_cropped.png,inceptionv3_roberts_results,roberts,0.002582,0.997418,1
17,Whole_namibia_cropped.png,inceptionv3_roberts_results,roberts,0.538836,0.461164,0


In [44]:
external_results_df.to_excel('External-Australia-Namibia-Predicitons.xlsx')